In [1]:
# import nest_asyncio
# nest_asyncio.apply()

import asyncio

In [2]:
import numpy as np
import time

In [3]:
import pyearthtools.data as petdata
from pyearthtools.data.exceptions import DataNotFoundError
import pyearthtools.pipeline as petpipe
import site_archive_jasmin # Required to not get missing something error

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/ssde/j25a/mmh_storage/theme3/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0Radar'}


In [4]:
himawari = petdata.archive.Himawari('surface_global_irradiance')
satpipe = petpipe.Pipeline(
    himawari
)


In [5]:
fullsat = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='1 day'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [6]:
async def load_sample(pipeline, sample_id):
    def sample():
        return pipeline[sample_id] # Unique per thread
        # return np.array([1, 2, 3, 4]) # Doesn't crash kernel when just returning a simple np array
    
    try:
        task_start_time = time.time()

        blocking_coroutine = asyncio.to_thread(sample)
        sample_task = asyncio.create_task(blocking_coroutine)
        
        sample = await sample_task
        
        print(f"Sample load time {sample_id} {time.time() - task_start_time}s")

        return sample
    except DataNotFoundError:
        return None
    except Exception as e:
        raise e

In [7]:
async def get_next_batch(pipeline, sample_id_iterator, sample_ix, max_samples, batch_size):
    tasks = set()
    batch = []
    while True:
        ########################### Adding sample task ###########################
        try:
            sample_id = next(sample_id_iterator)
        except StopIteration: # If any remaining in batch, don't process
            break

        tasks.add(asyncio.create_task(load_sample(pipeline, sample_id)))

        ########################### Checking if full samples ###########################
        if len(batch) + len(tasks) == batch_size:
            finished, unfinished = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
            print(f"{len(finished)} finished tasks, {len(unfinished)} unfinished tasks")

            for idx, task in enumerate(finished):
                sample = task.result()

                if sample is None:
                    continue

                if torch.isnan(torch.tensor(sample)).any():
                    continue

                batch.append(sample[0])
                sample_ix += 1

            print(f"Tasks length {len(tasks)}")
            tasks = unfinished
            print(f"Tasks length again{len(tasks)}")

            # The batch has been populated with the requird number of samples
            if len(batch) == batch_size:
                break # Break to outer scope & return
        
    return np.stack(batch)

In [ ]:
sample_date_iterator = iter(fullsat.iteration_order)

task_start_time = time.time()

next_batch_task = asyncio.create_task(get_next_batch(fullsat, sample_date_iterator, 0, 1e9, batch_size=2))
batch = await next_batch_task

batch_load_time = time.time() - task_start_time

print(f"Load  time: {batch_load_time:.2f}")
